# N010 · 调试、断言与可复现执行

**目标：** 定位报错并用小测试固定修复结果。

**先修：** N001, N004, N005。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** traceback；assert；unittest；随机种子；内核污染；最小复现。

**配套讲解来源：** [同名逐章意见](../../comment/010_debugging_reproducible_notebooks.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的是每个写代码的人都会遇到的两件事：**代码报错了怎么科学地定位（调试），以及修好之后怎么用小测试把"修好了"这件事钉死，保证以后重跑、换环境也不会悄悄坏掉（可复现）。**

本章借一道经典入门题当载体：**一维数组的运行总和（running sum）**——给你一个数组，输出一个新数组，新数组的第 i 项等于原数组前 i+1 项的和。

给你一个具体到数字的例子：

- 输入：`nums = [1, 2, 3]`
- 输出：`[1, 3, 6]`
- 为什么是它？第 0 项是前 1 个数的和 1；第 1 项是前 2 个数的和 1+2=3；第 2 项是前 3 个数的和 1+2+3=6。

而本章真正的教学重点是"怎么保证这个函数可靠"：比如你不小心写出了 `total` 定义在函数外面（全局变量）的版本，第一次调用没问题，第二次调用结果就会把上次的 total 叠加进去，越跑越错——这就是"重跑 Notebook 会不会被上次状态污染"的问题。解法就是本章代码里做的：**total 和输出列表都在函数内部重新初始化**，每次调用都从干净状态出发。

把"污染版"的翻车现场演一遍，你会更有感觉：

- 第一次调用 `running_sum([1,2,3])` 得到 `[1,3,6]`，一切正常。
- 但若 total 是全局的且没清零，第二次调用 `running_sum([1,2,3])` 就从 6 起步，得到 `[7,9,12]`。
- 同一段代码、同一个输入，两次结果不同——这就是"不可复现"，也是本章要用测试和局部状态堵死的洞。

## 2. 基础知识：先读懂这些词

- **traceback（回溯信息）**：程序报错时 Python 打印的那一大串"从哪行调用到哪行出错"的记录。读法是**从最后一行往上读**：最后一行告诉你错误类型（比如 IndexError），往上找第一个属于你自己代码的行，那就是出事现场。
- **assert（断言）**：`assert 条件` 的意思是"我断定这个条件成立，不成立就立刻报错"。我们用它是因为它能把"我期望的行为"写成可执行的检查——以后任何人改坏了代码，断言会第一时间炸给他看，而不是等到线上才发现。
- **unittest**：Python 标准库里的正式测试框架（本章知识点导览级别提及）。assert 适合随手检查，unittest 适合组织成套的测试用例、统一跑统一报告。
- **随机种子（random seed）**：给随机数生成器设定一个"起手式"，设了种子之后，同一份随机代码每次跑出的随机序列完全一样。我们提它是因为涉及随机的测试若不定种子，今天过了明天挂，没法复现。
- **内核污染（kernel pollution）**：Notebook 的变量在 cell 之间一直活着，前一个 cell 留下的全局变量会悄悄影响后一个 cell 的行为。本章重点防御的就是它——函数若依赖外部残留的 total，重跑就会叠加旧值。
- **最小复现（minimal reproduction）**：把一个大 bug 缩到"用最小输入、最少代码就能重现"的程度。输入越小，出错位置越藏不住。调试第一步先固定一个"最小输入 + 期望输出"，再去找第一个出错的位置。
- **前缀和（prefix sum / running sum）**：从左到右一路累加得到的数组，第 i 项是原数组前 i+1 项的总和。它是很多题的基础预处理。
- **O(1) 辅助状态**：除了输出以外，算法只用了常数个变量（这里是 total 一个），不随输入规模增长。

## 3. 思路推导：从小例子开始

`running_sum` 的推导线：

- **Step 1：先写"笨办法"看清需求。** 需求是 output[i] = nums[0..i] 之和。最直白的写法是每个位置都重新 `sum(nums[:i+1])` 算一遍前缀——能对，但第 i 次求和要把前 i+1 个数全加一遍，总共要做大约 n²/2 次加法。
- **Step 2：发现重复计算。** output[2] = 1+2+3，output[3] = 1+2+3+4，前三个数的加法被做了两遍。相邻两个前缀和之间只差"新进来的一个数"。
- **Step 3：改成边走边累加。** 维护一个 total，每读一个新数就 total 加上它，此时 total 恰好就是"到当前为止的和"，直接 append 进输出。这样每个数只被加一次。
- **Step 4：把状态放进函数里。** total=0 和 output=[] 都写在函数体内，每次调用都重新初始化——这是本章的调试主题：保证重跑不受上次影响（可复现）。
- **Step 5：用断言固定行为。** 挑几个有代表性的输入（正常、单元素、空、含负数），把期望输出写成 assert，一劳永逸。

**手算演示（输入 `[1, -2, 4]`，正好是 “运行示例”部分 表格用的数据）：**

| i | 输入值 | 前缀和（读入后的 total） |
|---|--------|--------------------------|
| 0 | 1 | 1 |
| 1 | -2 | 1 + (-2) = -1 |
| 2 | 4 | -1 + 4 = 3 |

这张表就是 “运行示例”部分 用 `show_table(['i','输入值','前缀和'], ...)` 生成的状态追踪表。对应输出数组 `[1, -1, 3]`。你可以看到 total 的含义始终是"从头到当前位置这一段的和"。

顺带把"拿到一个报错该按什么顺序下手"的固定动作列出来，这就是本章标题里"调试"两个字的落地流程：

1. 先读 traceback 的最后一行，确认错误类型（IndexError、NameError、AssertionError 各有含义）。
2. 再往上找第一个指向你自己代码的行号，那里是出事现场。
3. 把现场缩小成最小复现：换最小的输入，看错误还在不在。
4. 用手算（不是猜）写下这个最小输入的期望输出。
5. 对比实际输出与期望输出，定位第一个不一致的位置。
6. 修好之后，把"最小输入 + 期望输出"固化成一条 assert，防止同类错误回潮。

## 4. 核心代码：running_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def running_sum(nums: list[int]) -> list[int]:
    total = 0
    output = []
    for x in nums:
        total += x
        output.append(total)
    return output
```

### 逐段读懂代码

“分段实现”部分 定义了一个函数和一个测试函数。

### 函数 1：`running_sum` —— 一遍累加求前缀和

```python
def running_sum(nums: list[int]) -> list[int]:
    total = 0
    output = []
    for x in nums:
        total += x
        output.append(total)
    return output
```

- `def running_sum(nums: list[int]) -> list[int]:`：签名里 `list[int]` 是类型标注，声明"进来是整数列表，出去也是整数列表"。它不强制检查类型，但让读代码的人一眼明白契约。
- `total=0`：累加器在**函数内部**清零。这一行是本章调试主题的关键——如果把它挪到函数外面（全局），第二次调用就会在上次的旧值上继续加，重跑 Notebook 结果全错。
- `output=[]`：输出列表同样每次调用新建，不复用任何外部列表。理由同上：保证每次调用互不干扰（可复现）。
- `for x in nums:`：从左到右逐个取元素。
- `total+=x`：累加。此刻 total 的新含义恰好是"nums 从开头到当前位置的和"。
- `output.append(total)`：把当前累计值收进输出。
- `return output`：返回完整的前缀和数组。空输入时循环不执行，返回 `[]`，天然正确。

### 函数 2：`test_running_sum` —— 把期望行为写成断言

```python
def test_running_sum():
    assert running_sum([1, 2, 3]) == [1, 3, 6]
    assert running_sum([7]) == [7]
    assert running_sum([]) == []
    assert running_sum([-2, 5, -1]) == [-2, 3, 2]
```

- 这是一个"零参数、只抛断言"的测试函数：调用它要么静默通过，要么在某条 assert 处炸出 AssertionError 并告诉你哪一行。这比肉眼看 print 输出可靠，因为期望值是事先手算写死的，代码改坏了立刻暴露。
- 四条断言分别覆盖：正常多元素、单元素、空数组、含负数——具体意图第六节逐条讲。

“运行示例”部分 的 `show_table` 依旧是展示工具：它把 `zip(a, running_sum(a))` 的逐项结果拼成 HTML 表格，也就是第三节那张手算表。

## 5. 分段实现：按顺序运行

**本章接口：** `running_sum(nums)`、`test_running_sum()`

### running_sum

In [1]:
def running_sum(nums: list[int]) -> list[int]:
    total = 0
    output = []
    for x in nums:
        total += x
        output.append(total)
    return output

### test_running_sum

In [2]:
def test_running_sum():
    assert running_sum([1, 2, 3]) == [1, 3, 6]
    assert running_sum([7]) == [7]
    assert running_sum([]) == []
    assert running_sum([-2, 5, -1]) == [-2, 3, 2]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,-2,4]
show_table(['i','输入值','前缀和'], [(i,x,y) for i,(x,y) in enumerate(zip(a,running_sum(a)))])

i,输入值,前缀和
0,1,1
1,-2,-1
2,4,3


## 7. 正确性、适用条件与复杂度

扫描到 i 时，total 等于 nums[0:i+1] 的和；每次只新增当前元素，因此与逐前缀求和一致。

**代价：** O(n) 时间，O(n) 输出空间，O(1) 辅助状态。

### 展开理解

**为什么 `running_sum` 是对的？** 我们说清楚 total 在每个时刻的含义：开始时 total=0，等于"空段的和"，这没毛病。之后每读一个新数 x，total 就加上 x——而"前 i 个数的和"加上"第 i+1 个数"，恰好就是"前 i+1 个数的和"。所以每轮循环结束时，total 恰好等于"从开头到当前元素这一段的和"，我们也是在那一刻把它收进 output 的。因此 output 的每一项都和"对该下标重新从头求和"的结果一致，只是我们省掉了重复计算。空数组时一次循环都不跑，返回空列表，也符合"没有元素就没有前缀和"的定义。

**可复现性为什么有保证？** 因为 total 和 output 都是函数局部变量，每次调用都重新出生、函数结束就消失，内核里不留任何残留。哪怕你在同一个 Notebook 里连续调用一百次，每次都是从 0 开始算，前后互不影响——这正是 “思路推导”部分 里"函数内部重新初始化，重跑 Notebook 才不会把上次状态叠加"这句话的代码落点。

**复杂度是多少？** 时间 O(n)：每个元素被取出一次、加一次、append 一次，共约 3n 次基本操作。对比笨办法：每个位置重新求和，n 个位置各加最多 n 次，大约 n²/2 次加法——n=10⁵ 时，笨办法约 50 亿次操作（要跑好几秒），而累加法只有约 30 万次操作（一瞬间）。空间上，输出必须装 n 个结果所以是 O(n)；辅助变量只有 total 一个，是 O(1)。

## 8. 单元测试：每个用例在检查什么

先看 `test_running_sum` 内部的四条（“分段实现”部分）：

- `running_sum([1,2,3])==[1,3,6]`：**正常值测试**。三个正数，1 / 1+2 / 1+2+3，验证基本累加逻辑。
- `running_sum([7])==[7]`：**边界值测试（单元素）**。只有一个数时前缀和就是它自己，验证循环只跑一轮时的行为。
- `running_sum([])==[]`：**边界值测试（空输入）**。空数组应得到空数组，防空输入崩溃。
- `running_sum([-2,5,-1])==[-2,3,2]`：**特殊值测试（含负数）**。-2 / -2+5=3 / 3-1=2，验证累加对负数同样成立，total 也会随负数变小。

再看 “自动测试”部分 里外围的三条：

- `test_running_sum()`：先完整跑一遍上面的测试包，确保四条全部通过。
- `assert running_sum([1,2,3]) == running_sum([1,2,3])`：**可复现性测试**。同一个输入连续调用两次，结果必须完全一致。这一条专门抓"全局 total 残留"那类 bug——如果 total 在函数外，第二次调用会得到 [1,4,10] 之类的叠加值，这里立刻炸。
- `a=[3,4]; running_sum(a); assert a==[3,4]`：**纯函数测试（不改输入）**。调用之后原数组必须原封不动。这防止实现为了省空间直接在输入数组上原地改写（那种写法会坑到调用方）。

把七条检查按类型归个档，你以后给自己写测试时照着这个清单挑：

| 检查内容 | 类型 | 它防的是哪类 bug |
|----------|------|------------------|
| `[1,2,3]→[1,3,6]` | 正常值 | 主逻辑算错 |
| `[7]→[7]`、`[]→[]` | 边界值 | 循环零轮/一轮时崩 |
| `[-2,5,-1]→[-2,3,2]` | 特殊值（负数） | 累加方向、初值假设错 |
| 同输入连调两次相等 | 可复现性 | 全局状态残留 |
| 调用后 `a==[3,4]` | 纯函数性 | 原地改写输入 |

In [4]:
test_running_sum()

assert running_sum([1, 2, 3]) == running_sum([1, 2, 3])

a = [3, 4]

running_sum(a)

assert a == [3, 4]

print('N010: 所有本章断言通过')

N010: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 修复错位下标。

**练习 2：** 设计不会因前一单元格变量而通过的测试。

<details>
<summary>展开思路提示（不是完整答案）</summary>

“练习”部分 的两道练习：

- **练习 1（修复错位下标）**：想象（或故意写出）一个循环里下标用错的版本，比如把 `output[i]` 写成 `output[i-1]`，或者 range 边界差一。提示按"最小复现"流程走：第一步，选一个最小输入如 `[1,2]`，手算期望 `[1,3]`；第二步，跑错误版本，找到第一个不符合期望的位置（traceback 若有 IndexError，从最后一行往上读，定位到你自己的那行代码）；第三步，修正后用 `test_running_sum()` 里的断言确认。边界输入别忘了 `[7]` 和 `[]`。
- **练习 2（设计不会因前一单元格变量而通过的测试）**：这是在考"测试独立性"。提示：先想清楚"假通过"长什么样——比如某个测试函数拿全局变量 `a` 当输入，而 `a` 恰好被上一个 cell 改成了能通过的值，于是哪怕你的函数是错的，测试也绿了。对策是：所有测试输入都写成函数字面量（如 `running_sum([5,6])==[5,11]`），不引用任何外部名字；再加一条"同输入连调两次结果相同"的断言防状态残留。先手算 `[5,6]` 的期望输出 `[5,11]`，再写进断言。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def running_sum(nums: list[int]) -> list[int]:
    total = 0
    output = []
    for x in nums:
        total += x
        output.append(total)
    return output

def test_running_sum():
    assert running_sum([1, 2, 3]) == [1, 3, 6]
    assert running_sum([7]) == [7]
    assert running_sum([]) == []
    assert running_sum([-2, 5, -1]) == [-2, 3, 2]

# 示例与回归测试
test_running_sum()

assert running_sum([1, 2, 3]) == running_sum([1, 2, 3])

a = [3, 4]

running_sum(a)

assert a == [3, 4]

print('N010: 所有本章断言通过')

N010: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **1480. Running Sum of 1d Array（一维数组的运行总和）**：这题就是 `running_sum` 的原题——练的是"一遍扫描、边走边累加"的前缀和基本功，以及把期望行为固化成断言的测试习惯，warmup 难度。

照 notebook 原文提醒一句：本章实现遵循教学契约，与该题核心思路一致，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。